In [13]:
%pip install tqdm pandas plotly nbformat statsmodels

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.8 MB 14.1 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.8 MB 17.5 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 21.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.0 MB ? eta -:--:--
   ------- -------------------------------- 7.3/41.0 MB 37.0 MB/s eta 0:00:01
   -------------- ------------------------- 14.7/41.0 MB 36.6 MB/s eta 0:00:01
   -------------------- ------------------- 21.5/41.0 MB 34.8 MB/s eta 0:00:01
   ---------------------------- ----------- 28.8/41.0 MB 34.2 MB/s eta 0:00:01
   ----------------------------------- ---- 36.4/41.0 MB 34.5 MB/s eta 0:00:01
   ---------------------------------------- 41.0/41.0 MB 32.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Утилиты

In [117]:
import json

okved_transcription = {}

with open("okved2.json", encoding="utf-8") as f:
    okved_transcription = json.loads(f.read())

# Парсинг данных

In [ ]:
TOTAL_COMPANIES = 20000
PAGE_SIZE = 2000

url = "https://bo.nalog.ru/advanced-search/organizations/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36"
}

data = []

In [28]:
import time
import requests
import tqdm

MAX_DELAY = 600
BASE_DELAY = 1

for i in tqdm.tqdm(list(range(int(TOTAL_COMPANIES / PAGE_SIZE))), desc="Fetching data"):
    params = {
        "address": "Москва",
        "allFieldsMatch": "false",
        "page": i,
        "size": PAGE_SIZE
    }

    retries = 0

    while True:
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)  # Per-request timeout
            if response.ok:
                _data = response.json()
                data.append(_data)
                break
            else:
                print(f"Request failed with status code: {response.status_code}")
                print(response.text)
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e}")

        delay = min(BASE_DELAY * (2 ** retries), MAX_DELAY)
        print(f"Retrying in {delay} seconds...")
        time.sleep(delay)
        retries += 1

Fetching data:  50%|█████     | 5/10 [00:08<00:07,  1.58s/it]

Request failed with status code: 500
{"timestamp":"2025-02-21T08:57:59.547+0000","status":500,"error":"Internal Server Error","message":"all shards failed","path":"/advanced-search/organizations/"}
Retrying in 1 seconds...
Request failed with status code: 500
{"timestamp":"2025-02-21T08:58:00.634+0000","status":500,"error":"Internal Server Error","message":"all shards failed","path":"/advanced-search/organizations/"}
Retrying in 2 seconds...
Request failed with status code: 500
{"timestamp":"2025-02-21T08:58:02.721+0000","status":500,"error":"Internal Server Error","message":"all shards failed","path":"/advanced-search/organizations/"}
Retrying in 4 seconds...


Fetching data:  50%|█████     | 5/10 [00:15<00:15,  3.09s/it]


KeyboardInterrupt: 

## Сохранение первоначального парсинга

In [ ]:
import json


with open("data.json", "w") as f:
    f.write(json.dumps(data))

In [2]:
import json

with open("data.json") as f:
    data = json.loads(f.read())

## Подгрузка контента (статистики по организациям)

In [3]:
content = []

for page in data:
    content.extend(page["content"])
    
print(len(content))

10000


In [52]:
import json

with open("content.json") as f:
    content = json.loads(f.read())

# Подготовка данных

## Очистка организаций без статуса

In [53]:
statuses = set()

for row in content:
    statuses.add(row["statusCode"])

print(statuses)

{'INACTIVE', 'REORGANIZATION_STAGE', 'LIQUIDATION_STAGE', 'ACTIVE'}


In [44]:
content = [row for row in content if row["statusCode"] != None]
print(len(content)) # 9929 => diff 71

9929


In [ ]:
content = [row for row in content if row["okved2"] != None]
print(len(content)) # 9832 => diff 168

9832


# Исследование данных

## 1. Статистика по ликвидации организаций с 2020 года

Гипотеза: Организации стали чаще закрываться после начала ковидного кризиса, а также ввиду иных геополитических ситуаций

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.DataFrame(content)
df['statusDate'] = pd.to_datetime(df['statusDate'])

# Фильтруем организации по статусам
statuses = ['INACTIVE', 'REORGANIZATION_STAGE', 'LIQUIDATION_STAGE', 'ACTIVE']

fig = go.Figure()

for status in statuses:
    if status == 'ACTIVE':
        continue
    status_df = df[df['statusCode'] == status]
    status_by_date = status_df.groupby('statusDate').size().reset_index(name='count')
    status_by_date['cumulative_count'] = status_by_date['count'].cumsum()
    
    fig.add_trace(go.Scatter(
        x=status_by_date['statusDate'],
        y=status_by_date['cumulative_count'],
        mode='lines',
        name=status
    ))

fig.update_layout(
    title='Накопительные организации по статусам',
    xaxis_title='Дата',
    yaxis_title='Накопительное количество',
    legend_title='Статус организации'
)

fig.show()


Вывод: с января 2021 года резко возрастает количество организаций, так или иначе находящихся в статусе ликвидации, реорганизации или ликвидации, что может подтверждать гипотезу

## 2. Исследование ОКВЭД по выручке

Цель: найти, по каким ОКВЭДам с наибольшим ростом и падением выручки больше всего

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

bfo_data = []
_content = content.copy()
for row in _content:
    for _bfo_entry in row["bfo"]:
        _bfo_entry["okved2"]=okved_transcription[row["okved2"]]
    bfo_data.extend(row["bfo"])

df = pd.DataFrame(bfo_data)

# Преобразуемзначения в gainSum в числовой формат, заменяя все некорректные на NaN
df['gainSum'] = pd.to_numeric(df['gainSum'], errors='coerce')

# Отфильтруем строки с ненулевыми значениями 'gainSum'
df = df[df['gainSum'].notnull()]

# Преобразуем 'period' в числовой формат (год)
df['period'] = pd.to_numeric(df['period'])

# Группируем данные по ОКВЭД и году, вычисляем суммарную выручку для каждой группы
df_grouped = df.groupby(['okved2', 'period'])['gainSum'].sum().reset_index()

# Группируем по ОКВЭД и суммируем выручку за все годы, чтобы выбрать топ-10 ОКВЭДов
df_grouped_okved_total = df_grouped.groupby('okved2')['gainSum'].sum().reset_index()

# Сортируем по сумме выручки и выбираем топ-10 ОКВЭДов
df_grouped_okved_total = df_grouped_okved_total.sort_values(by='gainSum', ascending=False).head(10)

# Фильтруем исходный DataFrame по топ-10 ОКВЭДам
top_10_okveds = df_grouped_okved_total['okved2']
df_grouped_top10 = df_grouped[df_grouped['okved2'].isin(top_10_okveds)]

fig = go.Figure()

for okved in top_10_okveds:
    df_okved = df_grouped_top10[df_grouped_top10['okved2'] == okved]
    fig.add_trace(go.Scatter(
        x=df_okved['period'],
        y=df_okved['gainSum'],
        mode='lines+markers',
        name=f'ОКВЭД {okved}',
        marker=dict(size=8),
    ))

fig.update_layout(
    title="Суммарная выручка по годам в разрезе топ-10 ОКВЭДов",
    xaxis_title="Год",
    yaxis_title="Суммарная выручка (gainSum)",
    template="plotly_dark",
    legend_title="ОКВЭД"
)

fig.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

bfo_data = []
_content = content.copy()
for row in _content:
    for _bfo_entry in row["bfo"]:
        _bfo_entry["okved2"]=okved_transcription[row["okved2"]]
    bfo_data.extend(row["bfo"])

df = pd.DataFrame(bfo_data)

df['gainSum'] = pd.to_numeric(df['gainSum'], errors='coerce')

df = df[df['gainSum'].notnull()]

df['period'] = pd.to_numeric(df['period'])

# Группируем данные по ОКВЭД и году, рассчитываем суммарную выручку для каждой группы
df_grouped = df.groupby(['okved2', 'period'])['gainSum'].sum().reset_index()

# Для каждого ОКВЭД находим разницу между выручкой на последний и первый год
df_pivot = df_grouped.pivot(index='okved2', columns='period', values='gainSum')

# Рассчитываем изменение выручки по годам для каждого ОКВЭД
df_pivot['change_in_revenue'] = df_pivot.iloc[:, -1] - df_pivot.iloc[:, 0]  # Разница между последним и первым годом

# Сортируем по величине изменения выручки и берем топ-10
top_10_okveds = df_pivot.sort_values(by='change_in_revenue').head(10)

# Фильтруем исходные данные для топ-10 ОКВЭДов
df_grouped_top10 = df_grouped[df_grouped['okved2'].isin(top_10_okveds.index)]

fig = go.Figure()

for okved in top_10_okveds.index:
    df_okved = df_grouped_top10[df_grouped_top10['okved2'] == okved]
    fig.add_trace(go.Scatter(
        x=df_okved['period'],
        y=df_okved['gainSum'],
        mode='lines+markers',
        name=f'ОКВЭД {okved}',
        marker=dict(size=8),
    ))

fig.update_layout(
    title="Тренд изменения выручки по годам для ОКВЭДов с наибольшим снижением выручки",
    xaxis_title="Год",
    yaxis_title="Выручка (gainSum)",
    template="plotly_dark",
    legend_title="ОКВЭД"
)

fig.show()

## 3. Изменение длины наименования 

Гипотеза: сейчас организации именуются короче, чем раньше

In [ ]:
org_df = pd.DataFrame([{
    'shortName': org['shortName'],
    'nameLength': len(org['shortName']),
    'startYear': pd.to_datetime(org['statusDate']).year
} for org in content])

# Построим график рассеяния с трендовой линией
fig2 = px.scatter(org_df, x='startYear', y='nameLength', trendline='ols',
                  title='Тренд: Длина наименования vs Год начала деятельности')
fig2.update_traces(marker=dict(opacity=0))
fig2.update_layout(xaxis_title='Год начала деятельности', yaxis_title='Длина наименования')

# Показываем график
fig2.show()

Вывод: гипотеза подтвердилась, наблюдается снижение тренда длины наименования организаций

## 4. Анализ времени жизни организаций

In [129]:
import pandas as pd
import plotly.express as px
from datetime import datetime

current_date = datetime.now()

_data = []
for org in content:
    start_date = pd.to_datetime(org['statusDate'])
    end_date = current_date if org['statusCode'] == 'ACTIVE' else start_date
    lifespan = (end_date - start_date).days / 365.25  # Время жизни в годах
    _data.append({
        'shortName': org['shortName'],
        'lifespan_years': lifespan
    })

lifespan_df = pd.DataFrame(_data)

average_lifespan = lifespan_df['lifespan_years'].mean()
print(f"Среднее время жизни организаций: {average_lifespan:.2f} лет")

fig = px.histogram(lifespan_df, x='lifespan_years', nbins=int(lifespan_df['lifespan_years'].max()) + 1,
                   title='Распределение времени жизни организаций',
                   labels={'lifespan_years': 'Время жизни (лет)'})
fig.update_layout(bargap=0.1)
fig.show()

if average_lifespan < 3:
    print("Большинство организаций имеют короткий жизненный цикл, часто не превышающий 3 лет.")
elif average_lifespan < 7:
    print("Организации в среднем существуют от 3 до 7 лет, что указывает на умеренную устойчивость.")
else:
    print("Организации демонстрируют высокую устойчивость, с продолжительностью жизни более 7 лет.")

Среднее время жизни организаций: 11.33 лет


Организации демонстрируют высокую устойчивость, с продолжительностью жизни более 7 лет.


In [133]:
import pandas as pd
import plotly.express as px
from datetime import datetime
import math

name_gain_data = []
for org in content:
    name_length = len(org['shortName'])
    gain = np.average([bfo['gainSum'] for bfo in org['bfo'] if bfo['gainSum']])
    name_gain_data.append({'name_length': name_length, 'gainSum': gain})

name_gain_df = pd.DataFrame(name_gain_data)

fig = px.scatter(name_gain_df, x='name_length', y='gainSum', trendline='ols',
                 title='Корреляция между длиной наименования и выручкой',
                 labels={'name_length': 'Длина наименования', 'gainSum': 'Выручка'})
fig.update_traces(marker=dict(opacity=0))
fig.show()

Вывод: присутствует отрицательная корреляция между длиной наименования организации и её средней выручкой

## 5. Топ-100 компаний с наибольшей выручкой

In [ ]:
def calculate_average_revenue(org):
    total_revenue = sum(item['gainSum'] for item in org['bfo'] if item['gainSum'])
    periods = len(org['bfo'])
    return total_revenue / periods if periods else 0

organizations_with_avg_revenue = [
    {
        "id": org['id'],
        "shortName": org['shortName'],
        "averageRevenue": calculate_average_revenue(org),
        "okved2": org['okved2']
    }
    for org in content
]

sorted_organizations = sorted(organizations_with_avg_revenue, key=lambda x: x['averageRevenue'], reverse=True)

top_100_organizations = sorted_organizations[:100]

for idx, org in enumerate(top_100_organizations, 1):
    print(f"{idx}. {org['shortName']} - Средняя выручка: {org['averageRevenue']} тыс. руб. – {okved_transcription[org['okved2']]}")

1. АО "ЛАКСА ТРЕЙДИНГ" - Средняя выручка: 19029716.75 тыс. руб. – Торговля оптовая ювелирными изделиями
2. АО "ПНК" - Средняя выручка: 11757871.4 тыс. руб. – Добыча декоративного и строительного камня, известняка, гипса, мела и сланцев
3. ООО "ТГС" - Средняя выручка: 8373555.2 тыс. руб. – Работы строительные специализированные прочие, не включенные в другие группировки
4. АО "МТТ" - Средняя выручка: 8369358.0 тыс. руб. – Деятельность в области связи на базе проводных технологий
5. ООО "КРЕУСС" - Средняя выручка: 6570222.2 тыс. руб. – Торговля оптовая бытовыми электротоварами
6. ООО "СПЕЦДОРПРОЕКТ" - Средняя выручка: 6474629.2 тыс. руб. – Деятельность по эксплуатации автомобильных дорог и автомагистралей
7. АО "ИОНООБМЕННЫЕ ТЕХНОЛОГИИ" - Средняя выручка: 6156598.6 тыс. руб. – Торговля оптовая неспециализированная
8. АО "ТЛС" - Средняя выручка: 5206522.2 тыс. руб. – Аренда и лизинг железнодорожного транспорта и оборудования
9. ООО "ФОРБУК" - Средняя выручка: 5061067.4 тыс. руб. – Торговл